# Compound Signal Counterfactual

Our original counterfactual data only changes one demographic signal at a time. The proposal also asked: what happens when multiple signals change at once? Does the bias add up, get smaller, or stay about the same?

In this notebook we build a new counterfactual for each base resume where the name, the pronouns, and the university are all changed at the same time. We score them with SBERT and compare to the original.

In [ ]:
!pip install sentence-transformers pandas scikit-learn

In [ ]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

In [ ]:
from google.colab import files
uploaded = files.upload()
for filename in uploaded.keys():
    os.rename(filename, f"data/{filename}")
print("Files uploaded.")

In [ ]:
jobs = pd.read_csv("data/jobs.csv")
resumes = pd.read_csv("data/resume_variants.csv")
print("Jobs:", jobs.shape, "Resumes:", resumes.shape)

In [ ]:
# Pull the counterfactual values we already have for each resume_id and
# combine them into a single all-signals-changed version.
originals = resumes[resumes["version"] == "original"].set_index("resume_id")
name_cf = resumes[resumes["version"] == "name_changed"].set_index("resume_id")
pronoun_cf = resumes[resumes["version"] == "pronoun_changed"].set_index("resume_id")
uni_cf = resumes[resumes["version"] == "university_changed"].set_index("resume_id")

compound_rows = []
for rid in originals.index:
    orig = originals.loc[rid]
    new_name = name_cf.loc[rid, "name"]
    new_pronouns = pronoun_cf.loc[rid, "pronouns"]
    new_uni = uni_cf.loc[rid, "university"]

    text = str(orig["resume_text"])
    text = text.replace(str(orig["name"]), str(new_name))
    text = text.replace(str(orig["pronouns"]), str(new_pronouns))
    text = text.replace(str(orig["university"]), str(new_uni))

    compound_rows.append({
        "resume_id": rid,
        "domain": orig["domain"],
        "version": "all_changed",
        "changed_signal": "name+pronoun+university",
        "name": new_name,
        "pronouns": new_pronouns,
        "university": new_uni,
        "resume_text": text,
    })

compound = pd.DataFrame(compound_rows)
display(compound[["resume_id", "name", "pronouns", "university"]])

In [ ]:
def make_job_text(row):
    return f"{row['title']} {row['domain']} {row['company_name']} {row['job_description']}"

jobs["job_text"] = jobs.apply(make_job_text, axis=1)

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
job_embeddings = model.encode(jobs["job_text"].tolist())

orig_embeddings = model.encode(originals["resume_text"].tolist())
compound_embeddings = model.encode(compound["resume_text"].tolist())
print("Done embedding.")

In [ ]:
# Score every (original, job) and every (compound, job) pair.
orig_ids = list(originals.index)
comp_ids = list(compound["resume_id"])

rows = []
for i, rid in enumerate(orig_ids):
    for j, job_row in jobs.reset_index(drop=True).iterrows():
        orig_score = float(cosine_similarity(
            orig_embeddings[i].reshape(1, -1), job_embeddings[j].reshape(1, -1)
        )[0][0])
        comp_idx = comp_ids.index(rid)
        comp_score = float(cosine_similarity(
            compound_embeddings[comp_idx].reshape(1, -1), job_embeddings[j].reshape(1, -1)
        )[0][0])
        rows.append({
            "resume_id": rid,
            "job_id": job_row["job_id"],
            "job_title": job_row["title"],
            "original_score": orig_score,
            "compound_score": comp_score,
            "score_difference": comp_score - orig_score,
            "absolute_difference": abs(comp_score - orig_score),
        })

compound_comparison = pd.DataFrame(rows)
print("Comparison rows:", len(compound_comparison))
display(compound_comparison.head())

In [ ]:
compound_summary = pd.DataFrame({
    "changed_signal": ["name+pronoun+university"],
    "average_score_difference": [compound_comparison["score_difference"].mean()],
    "average_absolute_difference": [compound_comparison["absolute_difference"].mean()],
    "max_absolute_difference": [compound_comparison["absolute_difference"].max()],
})
display(compound_summary)

## How to read this against the single-signal results

Take the average absolute differences from the regular fairness summary (one row each for name, pronoun, university). Add them. If the compound average is close to that sum, the bias roughly adds up. If it is bigger, multiple signals reinforce each other. If it is smaller, they partly cancel out.

In [ ]:
compound.to_csv("results/compound_resume_variants.csv", index=False)
compound_comparison.to_csv("results/compound_comparison.csv", index=False)
compound_summary.to_csv("results/compound_summary.csv", index=False)
print("Saved.")

In [ ]:
from google.colab import files
files.download("results/compound_resume_variants.csv")
files.download("results/compound_comparison.csv")
files.download("results/compound_summary.csv")